# 04 — Canonical Gold referral model and snapshots

Build Gold referral facts and KPI views from the current Silver state. When
`AS_OF_DATE` is supplied by the archive replay notebook, calculations and the
snapshot use that historical export date. With a blank parameter, the notebook
uses the current date for the live pipeline.


In [ ]:
AS_OF_DATE = ""  # Optional YYYY-MM-DD; archive replay passes the month-end export date.
GOLD_SCHEMA = "gold"
SNAPSHOT_TABLE = "gold.fact_referral_snapshot"


In [ ]:
from datetime import date, datetime
from delta.tables import DeltaTable
from pyspark.sql import functions as F

if AS_OF_DATE:
    AS_OF_DATE_VALUE = datetime.strptime(AS_OF_DATE, "%Y-%m-%d").date()
else:
    AS_OF_DATE_VALUE = date.today()
AS_OF_SQL = f"DATE '{AS_OF_DATE_VALUE.isoformat()}'"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}")
print(f"Gold as-of date: {AS_OF_DATE_VALUE}")


In [ ]:
spark.sql("""
CREATE TABLE IF NOT EXISTS gold.cfg_placement_urgency_rule (
  PlacementUrgencyBand STRING, MaximumTargetDays INT,
  WarningHoursBeforeTarget INT, SortOrder INT, IsActive BOOLEAN
) USING DELTA
""")
spark.sql("""
MERGE INTO gold.cfg_placement_urgency_rule AS t
USING (
  SELECT * FROM VALUES
    ('Critical', 1, 6, 1, true), ('High', 3, 24, 2, true),
    ('Medium', 7, 48, 3, true), ('Planned', 99999, 72, 4, true),
    ('Unspecified', 99999, 72, 5, true)
  AS v(PlacementUrgencyBand, MaximumTargetDays, WarningHoursBeforeTarget, SortOrder, IsActive)
) AS s
ON t.PlacementUrgencyBand = s.PlacementUrgencyBand
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")


In [ ]:
spark.sql(f"""
CREATE OR REPLACE VIEW gold.fact_referral AS
WITH referral_history AS (
  SELECT *, ROW_NUMBER() OVER (
    PARTITION BY referral_id
    ORDER BY COALESCE(modified_timestamp, created_timestamp) DESC, rev DESC
  ) AS row_number_current
  FROM silver.slv_referral_aud
),
referral_current AS (
  SELECT * FROM referral_history WHERE row_number_current = 1
),
referral_created AS (
  SELECT referral_id, MIN(created_timestamp) AS ReferralCreatedDate
  FROM silver.slv_referral_aud GROUP BY referral_id
),
offer_rollup AS (
  SELECT rp.referral_id,
    MIN(o.offer_date) AS FirstOfferDate,
    MIN(CASE WHEN LOWER(o.offer_status) IN ('accepted','approved','selected')
        THEN o.last_modified_date END) AS OfferAcceptedDate,
    COUNT(DISTINCT o.offer_id) AS OfferCount,
    COUNT(DISTINCT o.provider_home_id) AS UniqueHomesOffered,
    MAX(COALESCE(o.last_modified_date, o.offer_date)) AS LastOfferActivityDate
  FROM silver.slv_offer o
  INNER JOIN silver.slv_referral_provider rp
    ON o.referral_provider_id = rp.referral_provider_id
  GROUP BY rp.referral_id
),
ipa_rollup AS (
  SELECT referral_id, MIN(created_datetime) AS IPAIssuedDate,
    MIN(placement_admission_date) AS PlannedPlacementStartDate,
    SUM(costs_total_weekly_fee) AS EstimatedWeeklyCost,
    MAX(COALESCE(updated_datetime, created_datetime)) AS LastIPAActivityDate
  FROM silver.slv_ipa GROUP BY referral_id
),
event_rollup AS (
  SELECT referral_id, MIN(event_timestamp) AS FirstActionDate,
    MAX(COALESCE(event_timestamp, created_timestamp)) AS LastEventActivityDate
  FROM silver.slv_referral_event_log GROUP BY referral_id
),
base AS (
  SELECT r.referral_id AS ReferralID, c.ReferralCreatedDate,
    r.required_start_date AS RequiredPlacementDate,
    r.response_required_by_date AS ResponseRequiredDate,
    r.modified_timestamp AS ReferralModifiedTimestamp,
    r.status AS CurrentStatus, r.placement_type_code AS PlacementTypeRequired,
    e.FirstActionDate, o.FirstOfferDate, o.OfferAcceptedDate, i.IPAIssuedDate,
    CASE WHEN LOWER(COALESCE(r.status, '')) IN ('closed','cancelled','withdrawn','completed')
      THEN r.modified_timestamp END AS ReferralClosedDate,
    CAST(NULL AS STRING) AS ReferralClosureReason,
    GREATEST(COALESCE(r.modified_timestamp, r.created_timestamp),
      e.LastEventActivityDate, o.LastOfferActivityDate, i.LastIPAActivityDate) AS LastActivityDate,
    o.OfferCount, o.UniqueHomesOffered,
    i.PlannedPlacementStartDate, i.EstimatedWeeklyCost,
    CASE
      WHEN r.required_start_date IS NULL THEN 'Unspecified'
      WHEN DATEDIFF(r.required_start_date, TO_DATE(c.ReferralCreatedDate)) <= 1 THEN 'Critical'
      WHEN DATEDIFF(r.required_start_date, TO_DATE(c.ReferralCreatedDate)) <= 3 THEN 'High'
      WHEN DATEDIFF(r.required_start_date, TO_DATE(c.ReferralCreatedDate)) <= 7 THEN 'Medium'
      ELSE 'Planned'
    END AS PlacementUrgencyBand
  FROM referral_current r
  INNER JOIN referral_created c ON r.referral_id = c.referral_id
  LEFT JOIN offer_rollup o ON r.referral_id = o.referral_id
  LEFT JOIN ipa_rollup i ON r.referral_id = i.referral_id
  LEFT JOIN event_rollup e ON r.referral_id = e.referral_id
)
SELECT {AS_OF_SQL} AS AsOfDate,
  ReferralID, ReferralCreatedDate, RequiredPlacementDate, ResponseRequiredDate,
  FirstActionDate, FirstOfferDate, OfferAcceptedDate, IPAIssuedDate,
  ReferralClosedDate, ReferralClosureReason, LastActivityDate, CurrentStatus,
  PlacementTypeRequired, PlacementUrgencyBand,
  CAST(NULL AS STRING) AS ChildCriticalityCode,
  COALESCE(OfferCount, 0) AS OfferCount,
  COALESCE(UniqueHomesOffered, 0) AS UniqueHomesOffered,
  COALESCE(OfferCount, 0) > 0 AS HasOffer,
  DATEDIFF(TO_DATE(FirstActionDate), TO_DATE(ReferralCreatedDate)) AS DaysToFirstAction,
  DATEDIFF(TO_DATE(FirstOfferDate), TO_DATE(ReferralCreatedDate)) AS DaysToFirstOffer,
  DATEDIFF(TO_DATE(OfferAcceptedDate), TO_DATE(ReferralCreatedDate)) AS DaysToAcceptedOffer,
  DATEDIFF(TO_DATE(IPAIssuedDate), TO_DATE(ReferralCreatedDate)) AS DaysToIPA,
  DATEDIFF(COALESCE(TO_DATE(ReferralClosedDate), {AS_OF_SQL}),
    TO_DATE(ReferralCreatedDate)) AS DaysOpen,
  DATEDIFF({AS_OF_SQL}, TO_DATE(LastActivityDate)) AS DaysWithoutActivity,
  CASE WHEN RequiredPlacementDate IS NOT NULL AND RequiredPlacementDate < {AS_OF_SQL}
    THEN DATEDIFF({AS_OF_SQL}, RequiredPlacementDate) ELSE 0 END AS DaysPastRequiredDate,
  LOWER(COALESCE(CurrentStatus, '')) NOT IN
    ('closed','cancelled','withdrawn','completed') AS IsOpen,
  IPAIssuedDate IS NOT NULL AND RequiredPlacementDate IS NOT NULL
    AND TO_DATE(IPAIssuedDate) <= RequiredPlacementDate AS PlacedByRequiredDate,
  CASE
    WHEN IPAIssuedDate IS NOT NULL AND RequiredPlacementDate IS NOT NULL
      AND TO_DATE(IPAIssuedDate) <= RequiredPlacementDate THEN 'Placed by target'
    WHEN IPAIssuedDate IS NOT NULL THEN 'Placed after target'
    WHEN RequiredPlacementDate < {AS_OF_SQL} AND LOWER(COALESCE(CurrentStatus, '')) NOT IN
      ('closed','cancelled','withdrawn','completed') THEN 'Open overdue'
    WHEN LOWER(COALESCE(CurrentStatus, '')) NOT IN
      ('closed','cancelled','withdrawn','completed') THEN 'Open on track'
    ELSE 'Closed without placement'
  END AS RequiredPlacementDateOutcome,
  PlannedPlacementStartDate, EstimatedWeeklyCost,
  CURRENT_TIMESTAMP() AS GoldModelledAt
FROM base
WHERE TO_DATE(ReferralCreatedDate) <= {AS_OF_SQL}
""")


In [ ]:
snapshot = spark.table("gold.fact_referral").select(
    F.lit(AS_OF_DATE_VALUE).cast("date").alias("SnapshotDate"),
    "ReferralID", "CurrentStatus", "PlacementUrgencyBand", "RequiredPlacementDate",
    "IsOpen", "HasOffer", "OfferCount", "DaysOpen", "DaysWithoutActivity",
    "DaysPastRequiredDate", "PlacedByRequiredDate", "RequiredPlacementDateOutcome",
)
if not spark.catalog.tableExists(SNAPSHOT_TABLE):
    snapshot.write.format("delta").mode("overwrite").saveAsTable(SNAPSHOT_TABLE)
else:
    target = DeltaTable.forName(spark, SNAPSHOT_TABLE)
    (target.alias("t").merge(snapshot.alias("s"),
        "t.SnapshotDate = s.SnapshotDate AND t.ReferralID = s.ReferralID")
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
print(f"Snapshot refreshed for {AS_OF_DATE_VALUE}: {snapshot.count():,} referrals")


In [ ]:
spark.sql("""
CREATE OR REPLACE VIEW gold.fact_referral_event_log AS
SELECT event_id AS EventID, referral_id AS ReferralID, event_type AS EventType,
  event_timestamp AS EventTimestamp, sequence_number AS SequenceNumber,
  created_by AS CreatedBy
FROM silver.slv_referral_event_log
""")
spark.sql("""
CREATE OR REPLACE VIEW gold.vw_kpi_referral_board_summary AS
SELECT AsOfDate, PlacementUrgencyBand, RequiredPlacementDateOutcome,
  COUNT(DISTINCT ReferralID) AS ReferralCount,
  SUM(CASE WHEN IsOpen THEN 1 ELSE 0 END) AS OpenReferralCount,
  SUM(CASE WHEN IsOpen AND RequiredPlacementDate < AsOfDate THEN 1 ELSE 0 END) AS OpenOverdueCount,
  SUM(CASE WHEN PlacedByRequiredDate THEN 1 ELSE 0 END) AS PlacedByRequiredDateCount,
  SUM(CASE WHEN HasOffer THEN 1 ELSE 0 END) AS ReferralsWithOfferCount,
  PERCENTILE_APPROX(DaysToIPA, 0.5) AS MedianDaysToIPA,
  SUM(COALESCE(EstimatedWeeklyCost, 0)) AS EstimatedWeeklyCost
FROM gold.fact_referral
GROUP BY AsOfDate, PlacementUrgencyBand, RequiredPlacementDateOutcome
""")
spark.sql("""
CREATE OR REPLACE VIEW gold.vw_kpi_referral_monthly AS
SELECT DATE_TRUNC('month', ReferralCreatedDate) AS ReferralCreatedMonth,
  COUNT(DISTINCT ReferralID) AS NewReferralCount,
  SUM(CASE WHEN HasOffer THEN 1 ELSE 0 END) AS ReferralsWithOfferCount,
  SUM(CASE WHEN IPAIssuedDate IS NOT NULL THEN 1 ELSE 0 END) AS IPACount,
  SUM(CASE WHEN PlacedByRequiredDate THEN 1 ELSE 0 END) AS PlacedByRequiredDateCount,
  SUM(CASE WHEN IsOpen THEN 1 ELSE 0 END) AS OpenReferralCount
FROM gold.fact_referral
GROUP BY DATE_TRUNC('month', ReferralCreatedDate)
""")
spark.sql("""
CREATE OR REPLACE VIEW gold.vw_provider_offer_performance AS
SELECT rp.provider_id AS ProviderID,
  COUNT(DISTINCT rp.referral_id) AS ReferralsReceived,
  COUNT(DISTINCT o.offer_id) AS OffersSubmitted,
  COUNT(DISTINCT CASE WHEN LOWER(o.offer_status) IN ('accepted','approved','selected')
    THEN o.offer_id END) AS OffersAccepted,
  COUNT(DISTINCT CASE WHEN f.PlacedByRequiredDate THEN f.ReferralID END) AS ReferralsPlacedByTarget
FROM silver.slv_referral_provider rp
LEFT JOIN silver.slv_offer o ON rp.referral_provider_id = o.referral_provider_id
LEFT JOIN gold.fact_referral f ON rp.referral_id = f.ReferralID
GROUP BY rp.provider_id
""")
